# TD2 - Diagnosing overfitting and leakage, and validating properly

**Machine Learning and AI for Analytical Chemistry - Part 3 (Evaluation)**

This tutorial makes the failure modes from the lecture concrete. You will:
- see overfitting as a train-to-test gap and control it with tree depth,
- create and then detect an information leak,
- compare a naive single split with proper (and grouped) cross-validation,
- standardise preprocessing with a pipeline so it cannot leak.

Dataset: the breast-cancer diagnostic dataset that ships with scikit-learn (30
quantitative features from digitised images of cell nuclei, a two-class target).
It is tabular, real, and needs no download. It is deliberately NOT the wine
dataset from TD1.

Runs as-is on Google Colab. Run cells top to bottom.


## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GroupKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score
RANDOM_STATE=42
np.random.seed(RANDOM_STATE)
data=load_breast_cancer(as_frame=True)
X=data.data; y=data.target
print("samples, features:", X.shape, " classes:", dict(y.value_counts()))


## 1. Overfitting as a train-to-test gap

We grow decision trees of increasing depth and plot training vs test accuracy.
The training curve keeps rising; the test curve peaks and then stalls or falls.
The gap between them is overfitting.

In [ ]:
X_tr,X_te,y_tr,y_te=train_test_split(X,y,test_size=0.3,random_state=RANDOM_STATE,stratify=y)
depths=range(1,16); tr=[]; te=[]
for d in depths:
    m=DecisionTreeClassifier(max_depth=d,random_state=RANDOM_STATE).fit(X_tr,y_tr)
    tr.append(accuracy_score(y_tr,m.predict(X_tr)))
    te.append(accuracy_score(y_te,m.predict(X_te)))
plt.plot(list(depths),tr,marker="o",label="train")
plt.plot(list(depths),te,marker="s",label="test")
plt.xlabel("max_depth"); plt.ylabel("accuracy"); plt.legend(); plt.title("Train vs test vs complexity"); plt.show()
print("best test depth:", list(depths)[int(np.argmax(te))])

## 2. Create a leak, then detect it

A common chemistry-style leak: replicate measurements of the same item appear in
both train and test, so the model recognises the item instead of generalising.
We simulate this by duplicating each sample (as a "replicate") with a tiny amount
of noise, then compare a naive random split (leaky) with a grouped split that
keeps replicates of the same item together.

In [ ]:
rng=np.random.default_rng(RANDOM_STATE)
# build a replicated dataset: 2 noisy replicates per original sample, sharing a group id
Xr=[]; yr=[]; groups=[]
for i in range(len(X)):
    for _ in range(2):
        Xr.append(X.iloc[i].values + rng.normal(0,0.01*np.abs(X.iloc[i].values)))
        yr.append(y.iloc[i]); groups.append(i)
Xr=np.array(Xr); yr=np.array(yr); groups=np.array(groups)

clf=RandomForestClassifier(n_estimators=200,random_state=RANDOM_STATE)
# leaky: random split ignores groups -> replicates of one item on both sides
leaky=cross_val_score(clf,Xr,yr,cv=StratifiedKFold(5,shuffle=True,random_state=RANDOM_STATE)).mean()
# honest: grouped split keeps all replicates of an item in one fold
honest=cross_val_score(clf,Xr,yr,cv=GroupKFold(5),groups=groups).mean()
print("leaky (random split)   accuracy:", round(leaky,3))
print("honest (grouped split) accuracy:", round(honest,3))
print("inflation from leakage :", round(leaky-honest,3))

## 3. Preprocessing that leaks vs a pipeline that does not

Scaling and feature selection must be fit on training data only. If you fit them
on the whole dataset before cross-validation, test information leaks in. The
correct way is to put them inside a Pipeline so each fold refits them.

In [ ]:
# WRONG: scale + select on ALL data, then cross-validate (leaks)
X_all_scaled=StandardScaler().fit_transform(X)
X_all_sel=SelectKBest(f_classif,k=10).fit_transform(X_all_scaled,y)
wrong=cross_val_score(SVC(kernel="rbf",random_state=RANDOM_STATE),X_all_sel,y,
                      cv=StratifiedKFold(5,shuffle=True,random_state=RANDOM_STATE)).mean()
# RIGHT: pipeline refits scaler + selector inside each fold
pipe=make_pipeline(StandardScaler(),SelectKBest(f_classif,k=10),SVC(kernel="rbf",random_state=RANDOM_STATE))
right=cross_val_score(pipe,X,y,cv=StratifiedKFold(5,shuffle=True,random_state=RANDOM_STATE)).mean()
print("leaky preprocessing  :", round(wrong,3))
print("pipeline (no leak)   :", round(right,3))

## 4. Reflection

1. In part 1, at which depth does the test curve stop improving, and what is the
   train-to-test gap at the deepest tree?
2. In part 2, how large was the accuracy inflation caused by the leak? Why does
   grouping remove it?
3. In part 3, why does fitting the scaler and feature selector on all the data
   inflate the score, even though you still cross-validate the classifier?
4. Which single habit from this notebook would most improve the trustworthiness
   of a retention-time model?

*Write your answers here.*
